In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install transformers datasets accelerate -q

from datasets import load_dataset
wiki_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [2]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

wiki_train_raw = wiki_dataset["train"].filter(lambda x: len(x["text"].strip()) > 20)
wiki_val_raw = wiki_dataset["validation"].filter(lambda x: len(x["text"].strip()) > 20)

wiki_train = wiki_train_raw.select(range(min(20000, len(wiki_train_raw))))
wiki_val = wiki_val_raw.select(range(min(2000, len(wiki_val_raw))))

wiki_tokenized_train = wiki_train.map(tokenize_fn, batched=True, remove_columns=["text"])
wiki_tokenized_val = wiki_val.map(tokenize_fn, batched=True, remove_columns=["text"])

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3760 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [3]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [4]:
from transformers import GPT2Config, GPT2LMHeadModel

tiny_config = GPT2Config(vocab_size=tokenizer.vocab_size, n_positions=128, n_embd=128, n_layer=4, n_head=4)
zoom_config = GPT2Config(vocab_size=tokenizer.vocab_size, n_positions=128, n_embd=144, n_layer=4, n_head=4)
xsmall_config = GPT2Config(vocab_size=tokenizer.vocab_size, n_positions=128, n_embd=160, n_layer=4, n_head=4)
small_config = GPT2Config(vocab_size=tokenizer.vocab_size, n_positions=128, n_embd=192, n_layer=6, n_head=6)
medium_config = GPT2Config(vocab_size=tokenizer.vocab_size, n_positions=128, n_embd=256, n_layer=8, n_head=8)

In [5]:
teacher_gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
teacher_gpt2.eval()
teacher_gpt2.to("cuda")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [6]:
import torch
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments

class DistillationTrainer(Trainer):
    def __init__(self, teacher_model=None, temperature=2.0, alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        self.teacher = teacher_model
        self.temperature = temperature
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        student_loss = outputs.loss
        student_logits = outputs.logits
        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)
            teacher_logits = teacher_outputs.logits
        T = self.temperature
        student_logits_flat = student_logits.view(-1, student_logits.size(-1))
        teacher_logits_flat = teacher_logits.view(-1, teacher_logits.size(-1))
        distill_loss = F.kl_div(
            F.log_softmax(student_logits_flat / T, dim=-1),
            F.softmax(teacher_logits_flat / T, dim=-1),
            reduction="batchmean"
        ) * (T ** 2)
        loss = self.alpha * student_loss + (1 - self.alpha) * distill_loss
        return (loss, outputs) if return_outputs else loss

def run_baseline_condition(config, seed, args_template):
    torch.manual_seed(seed)
    model = GPT2LMHeadModel(config)
    model.to("cuda")
    trainer = Trainer(
        model=model, args=args_template,
        train_dataset=wiki_tokenized_train, eval_dataset=wiki_tokenized_val, data_collator=data_collator,
    )
    trainer.train()
    result = trainer.evaluate()
    return result["eval_loss"]

def run_distilled_condition(config, teacher_model, seed, args_template, alpha=0.5, temperature=2.0):
    torch.manual_seed(seed)
    model = GPT2LMHeadModel(config)
    model.to("cuda")
    trainer = DistillationTrainer(
        model=model, teacher_model=teacher_model, temperature=temperature, alpha=alpha,
        args=args_template,
        train_dataset=wiki_tokenized_train, eval_dataset=wiki_tokenized_val, data_collator=data_collator,
    )
    trainer.train()
    plain_eval_trainer = Trainer(
        model=model, args=args_template,
        eval_dataset=wiki_tokenized_val, data_collator=data_collator,
    )
    result = plain_eval_trainer.evaluate()
    return result["eval_loss"]

In [7]:
wiki_results = {}
configs = {"Tiny": tiny_config, "Medium": medium_config}

for name, cfg in configs.items():
    args = TrainingArguments(
        output_dir=f"./wiki_{name}",
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=1, logging_steps=200, eval_strategy="steps",
        eval_steps=200, save_strategy="no", report_to="none",
    )
    base_loss = run_baseline_condition(cfg, seed=42, args_template=args)
    dist_loss = run_distilled_condition(cfg, teacher_gpt2, seed=42, args_template=args, alpha=0.5)
    wiki_results[name] = {"baseline": base_loss, "distilled": dist_loss, "gap": base_loss - dist_loss}
    print(f"{name}, WikiText-2: baseline={base_loss:.3f}, distilled={dist_loss:.3f}, gap={base_loss-dist_loss:.3f}")

print(wiki_results)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
200,9.886469,9.245744
400,8.869641,8.481949
600,8.263844,8.018905
800,7.906105,7.747368
1000,7.702423,7.608018
1200,7.596097,7.555820


Step,Training Loss,Validation Loss
200,8.403585,7.786225
400,7.423159,7.037132
600,6.822365,6.559710
800,6.439296,6.268125
1000,6.217575,6.105387
1200,6.088219,6.040236


Tiny, WikiText-2: baseline=7.554, distilled=7.760, gap=-0.206


Step,Training Loss,Validation Loss
200,9.132823,8.073412
400,7.618970,7.240476
600,7.148811,7.033104
800,7.028224,6.949170
1000,6.967133,6.903962
1200,6.914082,6.882484


Step,Training Loss,Validation Loss
200,7.776270,6.777083
400,6.291788,5.836851
600,5.651481,5.438138
800,5.371967,5.257536
1000,5.244296,5.167959
1200,5.159250,5.132058


Medium, WikiText-2: baseline=6.882, distilled=6.965, gap=-0.083
{'Tiny': {'baseline': 7.553736686706543, 'distilled': 7.7600226402282715, 'gap': -0.20628595352172852}, 'Medium': {'baseline': 6.881779193878174, 'distilled': 6.965246200561523, 'gap': -0.08346700668334961}}
